In [1]:
# # python-dotenv → manage API keys

# !pip install pymupdf pdfplumber pytesseract Pillow \
#              langchain langchain-community \
#              sentence-transformers faiss-cpu \
#              python-dotenv openai -q


In [2]:
# !pip list

## Chunking Based on Section using Pattern Recognition Pure Regex method

In [3]:
import camelot

tables = camelot.read_pdf(
    r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf",
    pages="16"
)

df = tables[0].df

headers=df.values.tolist()[0]
rows=df.values.tolist()[1:]

print(headers)
print(rows)
pairs = []
for row in rows:
    for header, value in zip(headers, row):
        pairs.append(f"{header}: {value}")

final_text = " | ".join(pairs)

d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\pypdf\_crypt_providers\_cryptography.py:32: CryptographyDeprecationWarning: ARC4 has been moved to cryptography.hazmat.decrepit.ciphers.algorithms.ARC4 and will be removed from cryptography.hazmat.primitives.ciphers.algorithms in 48.0.0.
  from cryptography.hazmat.primitives.ciphers.algorithms import AES, ARC4


['Particulars', 'Opening', 'Addition', 'Closing']
[['Capital Expenditure during the year', '7,334.69', '746.43', '8,081.12'], ['Capitalization', '3,987.92', '7.51', '3,995.42'], ['Capital Work in Progress', '3,346.78', '738.92', '4,085.70'], ['Asset Capital Work in Progress', '2,683.93', '712.96', '3,396.89'], ['Capital Advances', '635.84', '25.98', '661.82'], ['Advance to Suppliers', '0.00', '0.00', '0.00'], ['Stock of Materials at Site', '27.00', '-0.01', '26.99']]


In [4]:
final_text

'Particulars: Capital Expenditure during the year | Opening: 7,334.69 | Addition: 746.43 | Closing: 8,081.12 | Particulars: Capitalization | Opening: 3,987.92 | Addition: 7.51 | Closing: 3,995.42 | Particulars: Capital Work in Progress | Opening: 3,346.78 | Addition: 738.92 | Closing: 4,085.70 | Particulars: Asset Capital Work in Progress | Opening: 2,683.93 | Addition: 712.96 | Closing: 3,396.89 | Particulars: Capital Advances | Opening: 635.84 | Addition: 25.98 | Closing: 661.82 | Particulars: Advance to Suppliers | Opening: 0.00 | Addition: 0.00 | Closing: 0.00 | Particulars: Stock of Materials at Site | Opening: 27.00 | Addition: -0.01 | Closing: 26.99'

In [5]:
import fitz          # PyMuPDF
import pdfplumber
import json
import os
import io
from pathlib import Path
from PIL import Image

# Verify versions
print(f"PyMuPDF version  : {fitz.version}")
print(f"pdfplumber       : {pdfplumber.__version__}")
print("✅ Core imports OK")

PyMuPDF version  : ('1.26.7', '1.26.12', None)
pdfplumber       : 0.11.9
✅ Core imports OK


In [6]:
%pwd

'd:\\python scripts\\Generative AI\\arise_chatbot\\research'

In [7]:
import os
os.chdir("../")

In [8]:
# petition path
PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"
# This is what we discussed — document-level metadata
DOC_METADATA = {
    "discom": "JUSNL",
    "document_type": "petition",   # petition / tariff_order / regulation
    "filing_year": "FY26",         # the year the petition covers
    "commission": "JSERC",
    "source_file": PDF_PATH
}

In [9]:
# Check file exists
if Path(PDF_PATH).exists():
    print(f"✅ PDF found: {PDF_PATH}")
    doc = fitz.open(PDF_PATH)
    print(f"   Total pages: {len(doc)}")
    doc.close()
else:
    print(f"⚠️  PDF not found at: {PDF_PATH}")
    print("   Place your PDF in the data/petitions/ folder and update PDF_PATH above")

✅ PDF found: D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf
   Total pages: 66


In [10]:
def detect_page_type(page: fitz.Page) -> str:
    """
    Classify a PDF page as 'digital' or 'scanned'.

    Digital  → PyMuPDF can extract readable text
    Scanned  → Page is an image, text extraction returns nothing
    """
    text = page.get_text().strip()
    # print(text)

    # < 50 chars means either scanned or a blank/diagram page
    # Both cases need OCR to be useful
    if len(text) < 50:
        return "scanned"
    return "digital"


# ── TEST: Run classifier on first 10 pages of your PDF ──
if Path(PDF_PATH).exists():
    doc = fitz.open(PDF_PATH)
    print("Page | Type    | Chars extracted")
    print("-" * 40)
    for i in range(min(10, len(doc))):
        page = doc[i]
        ptype = detect_page_type(page)
        chars = len(page.get_text().strip())
        flag = "⚠️ " if ptype == "scanned" else "✅"
        print(f"  {i+1:2d} | {ptype:<8} | {chars:>5} chars  {flag}")
    doc.close()
else:
    print("⚠️  Add your PDF first (Cell 3)")

Page | Type    | Chars extracted
----------------------------------------
   1 | digital  |   386 chars  ✅
   2 | digital  |  2399 chars  ✅
   3 | digital  |  2471 chars  ✅
   4 | digital  |  1416 chars  ✅
   5 | digital  |  6049 chars  ✅
   6 | digital  |  2757 chars  ✅
   7 | digital  |  7398 chars  ✅
   8 | digital  |  2453 chars  ✅
   9 | digital  |  2991 chars  ✅
  10 | digital  |  2439 chars  ✅


In [11]:
def table_to_llm_text(df, table_index: int) -> str:
    """
    Convert camelot dataframe → clean LLM-readable text.
    Format: Table name + Row-by-row header:value pairs
    """
    rows = df.values.tolist()
    # print(rows)

    if len(rows) < 2:
        return ""

    headers = [str(h).strip() for h in rows[0]]
    # print(headers)
    data_rows = rows[1:]

    lines = [f"Table {table_index + 1}:"]
    # print(data_rows)

    for row_num, row in enumerate(data_rows, start=0):
        pairs = []
        for header, value in zip(headers, row):
            value = str(value).strip()
            # print(value)
            if value:  # skip empty cells
                pairs.append(f"{header}: {value}")
        if pairs:
            lines.append(f"  Row {row_num} → {' | '.join(pairs)}")

    return "\n".join(lines)
import re

def merge_text_and_tables(raw_text: str, tables_text: list[str]) -> str:
    """
    Merge paragraph text and tables in correct sequence.
    Uses table title lines as anchor points for insertion.
    """
    # Split text at table title lines e.g. "Table 3 Capital Expenditure..."
    segments = re.split(r'(Table\s+\d+[^\n]*)', raw_text)

    result = []
    table_idx = 0

    for segment in segments:
        # If this segment is a table title
        if re.match(r'Table\s+\d+', segment.strip()):
            # Insert the clean camelot table here
            if table_idx < len(tables_text):
                result.append(f"\n{segment.strip()}")  # keep title
                result.append(tables_text[table_idx])   # insert clean table
                table_idx += 1
        else:
            if segment.strip():
                result.append(segment.strip())

    return "\n\n".join(result)

def extract_digital_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text via pdfplumber + tables via camelot.
    Sequence preserved — text first, then tables.
    """
    # Text extraction
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[page_num]
        page_height = page.height

        # Get bounding boxes of all tables
        table_bboxes = [t.bbox for t in page.find_tables()]

        # Filter out table regions from text extraction
        raw_text = page.filter(
            lambda obj: (
                # condition 1: remove header and footer by position
                obj.get("top", 0) > 50 and
                obj.get("bottom", 0) < page_height - 50
            ) and (
                # condition 2: remove table regions
                not any(
                    obj["x0"] >= bbox[0] and
                    obj["top"] >= bbox[1] and
                    obj["x1"] <= bbox[2] and
                    obj["bottom"] <= bbox[3]
                    for bbox in table_bboxes
                )
            )
        ).extract_text() or ""

    # Table extraction
    tables = camelot.read_pdf(
        pdf_path,
        pages=str(page_num + 1),
        flavor="lattice",
        line_scale=40   # use 'stream' if tables have no borders
    )

    tables_as_text = []
    for i, table in enumerate(tables):
        text = table_to_llm_text(table.df, i)
        # print(text)
        if text:
            tables_as_text.append(text)



    merged_text = merge_text_and_tables(raw_text, tables_as_text)


    return {
        "text"        : merged_text,
        "tables_text" : tables_as_text,   # LLM-readable
        "tables_raw"  : [t.df for t in tables],  # dataframes for debugging
        "page_number" : page_num + 1,
        "page_type"   : "digital"
    }


# ── TEST: Extract a page that likely has a table ──
# Page 16 in JUSNL petition = Table 3 (Capex) — try this page
PAGE_NUM = 14   # 0-indexed → page 16 in PDF

# if Path(PDF_PATH).exists():
result = extract_digital_page(PDF_PATH, PAGE_NUM)
print(result['text'])  # print first 500 chars of merged text


    # print(f"=== Page {result['page_number']} ===")
    # print(f"Text length  : {len(result['text'])} chars")
    # print(f"Tables found : {len(result['tables_raw'])}")
    # print()
    # print("--- Text preview (first 400 chars) ---")
    # print(result['text'][:])
    # print()
    # if result['tables_text']:
    #     print("--- First table (cleaned for LLM) ---")
    #     print(result['tables_text'])
    # else:
    #     print("No tables found on this page — try a different TEST_PAGE number")
# else:
#     print("⚠️  Add your PDF first")

3. Provisional True up For FY 2023-24
3.1. Preamble
3.1.1. This section outlines the actual performance of the JUSNL during the FY 2023-
24.
3.1.2. In line with the provisions of the JSERC (Terms and Conditions for Determination
of Transmission Tariff) Regulations, 2020, the Petitioner hereby submits the
Provisional True up for FY 2023-24. The expenses of the Petitioner presented in
the true up are based on the unaudited books of accounts for the FY 2023-24.
The ARR so arrived has been compared with that approved by the Hon’ble
Commission vide its Business plan and ARR for MYT Period FY 2021-22 to FY
2025-26 Order dated 23rd June, 2023 and its Tariff Order dated 03.07.2024.
Accordingly, the Aggregate Revenue Requirement, revenue and gap for FY
2023-24 have been given in the subsequent sub-sections of this chapter.
3.2. Basis of True up
3.2.1. The unaudited annual account of JUSNL for FY 2023-24 and unaudited Trial
Balance of SLDC for FY 2023-24 are attached as Annexure A. The statutory

In [12]:
import re

# def is_main_section(line: str) -> bool:
#     """
#     Matches: 3.4. Gross Fixed Asset
#     Not:     3.4.1. The Commission
#     """
#     # ^\d+\.\d+\.   → starts with digit.digit.
#     # \s            → followed by space (not another digit)
#     pattern = r'^\d+\.\d+\.\s'
#     return bool(re.match(pattern, line.strip()))

# # Test


# # when you called extract_digital_page, store it as:
# page_result = extract_digital_page(PDF_PATH, PAGE_NUM)

# # then test chunking on it:
# for line in page_result['text'].splitlines():
#     resu = is_main_section(line)
#     print(f"{'✅' if resu else '❌'} {line}")

def is_main_section(line: str) -> bool:
    line = line.strip()

    # exclude TOC lines
    if '.....' in line:
        return False

    # exclude 4-digit years
    if re.match(r'^\d{4}\.', line):
        return False

    # exclude long lines
    if len(line) > 80:
        return False

    # exclude high numbers — real sections only 1-9
    first_num = re.match(r'^(\d+)\.', line)
    if first_num and int(first_num.group(1)) > 9:
        return False

    p1 = r'^\d+\.\s[A-Z]'
    p2 = r'^\d+\.\d+\.\s[A-Z]'

    return bool(re.match(p1, line)) or bool(re.match(p2, line))
# # then test chunking on it:
# for line in page_result['text'].splitlines():
#     resu = is_main_section(line)
#     print(f"{'✅' if resu else '❌'} {line}")

In [13]:
def chunk_page(text: str, page_metadata: dict,
               initial_heading: str = "introduction",
               initial_main_section: str = "introduction") -> list[dict]:

    chunks = []
    current_lines = []
    current_heading = initial_heading
    current_main_section = initial_main_section

    for line in text.splitlines():

        if is_main_section(line):
            # save current chunk if it has content
            if current_lines:
                chunk_text = " ".join(current_lines).strip()
                if len(chunk_text) > 100:
                    chunks.append({
                        "text"            : chunk_text,
                        "chunk_type"      : "text",
                        "section_heading" : current_heading,
                        "main_section"    : current_main_section,
                        **page_metadata
                    })

            # update main section if top level → "3. " not "3.1. "
            if re.match(r'^\d+\.\s[A-Z]', line.strip()):
                current_main_section = line.strip()

            # start new chunk
            current_heading = line.strip()
            current_lines = [line.strip()]  # include heading in chunk text

        else:
            current_lines.append(line.strip())

    # save last chunk — main_section was missing here before
    if current_lines:
        chunk_text = " ".join(current_lines).strip()
        if len(chunk_text) > 100:
            chunks.append({
                "text"            : chunk_text,
                "chunk_type"      : "text",
                "section_heading" : current_heading,
                "main_section"    : current_main_section,  # ← was missing
                **page_metadata
            })

    return chunks

In [14]:
page_metadata = {
    "page_number"  : PAGE_NUM+1,
    "discom"       : "JUSNL",
    "document_type": "petition",
    "filing_year"  : "FY26"
}

chunks = chunk_page(result['text'], page_metadata)

print(f"Total chunks: {len(chunks)}")
for i, c in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print(f"  Heading : {c['section_heading']}")
    print(f"  Text    : {c['text'][:]}")

Total chunks: 3

Chunk 1
  Heading : 3.1. Preamble
  Text    : 3.1. Preamble 3.1.1. This section outlines the actual performance of the JUSNL during the FY 2023- 24. 3.1.2. In line with the provisions of the JSERC (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020, the Petitioner hereby submits the Provisional True up for FY 2023-24. The expenses of the Petitioner presented in the true up are based on the unaudited books of accounts for the FY 2023-24. The ARR so arrived has been compared with that approved by the Hon’ble Commission vide its Business plan and ARR for MYT Period FY 2021-22 to FY 2025-26 Order dated 23rd June, 2023 and its Tariff Order dated 03.07.2024. Accordingly, the Aggregate Revenue Requirement, revenue and gap for FY 2023-24 have been given in the subsequent sub-sections of this chapter.

Chunk 2
  Heading : 3.2. Basis of True up
  Text    : 3.2. Basis of True up 3.2.1. The unaudited annual account of JUSNL for FY 2023-24 and unaudite

In [15]:
chunks

[{'text': '3.1. Preamble 3.1.1. This section outlines the actual performance of the JUSNL during the FY 2023- 24. 3.1.2. In line with the provisions of the JSERC (Terms and Conditions for Determination of Transmission Tariff) Regulations, 2020, the Petitioner hereby submits the Provisional True up for FY 2023-24. The expenses of the Petitioner presented in the true up are based on the unaudited books of accounts for the FY 2023-24. The ARR so arrived has been compared with that approved by the Hon’ble Commission vide its Business plan and ARR for MYT Period FY 2021-22 to FY 2025-26 Order dated 23rd June, 2023 and its Tariff Order dated 03.07.2024. Accordingly, the Aggregate Revenue Requirement, revenue and gap for FY 2023-24 have been given in the subsequent sub-sections of this chapter.',
  'chunk_type': 'text',
  'section_heading': '3.1. Preamble',
  'main_section': '3. Provisional True up For FY 2023-24',
  'page_number': 15,
  'discom': 'JUSNL',
  'document_type': 'petition',
  'fi

In [16]:
# Standard cost heads from Indian electricity regulations
COST_HEAD_KEYWORDS = {
    "capex"         : ["capital expenditure", "capex", "cwip", "capitalization",
                       "capital investment", "scheme", "project"],
    "employee_cost" : ["employee", "salary", "staff", "manpower", "hr",
                       "wages", "pension", "gratuity"],
    "om_expenses"   : ["operation and maintenance", "o&m", "repair",
                       "maintenance", "a&g", "administrative"],
    "depreciation"  : ["depreciation", "asset life", "useful life", "dep rate"],
    "interest"      : ["interest", "loan", "borrowing", "debt", "working capital",
                       "interest on loan"],
    "roe"           : ["return on equity", "roe", "equity", "normative equity"],
    "power_purchase": ["power purchase", "ppc", "energy charge", "capacity charge",
                       "ppa", "merit order", "short term purchase"],
    "revenue_gap"   : ["revenue gap", "regulatory asset", "carrying cost",
                       "unrecovered", "revenue requirement"],
    "tariff"        : ["tariff", "consumer category", "slab", "transmission charge",
                       "wheeling charge"],
    "true_up"       : ["true up", "true-up", "trueup", "actual vs approved",
                       "reconciliation", "provisional"],
    "arr"           : ["annual revenue requirement", "arr", "aggregate revenue"],
    "at_c_loss"     : ["at&c", "aggregate technical", "commercial loss", "transmission loss"]
}


def map_cost_head_keywords(text: str, heading: str) -> str:
    """
    Fast keyword-based cost head mapping.
    Check heading first (more reliable), then full text.
    Returns matched cost head or 'other'.
    """
    search_text = (heading + " " + text[:500]).lower()

    scores = {}
    for cost_head, keywords in COST_HEAD_KEYWORDS.items():
        score = sum(1 for kw in keywords if kw in search_text)
        if score > 0:
            scores[cost_head] = score

    if scores:
        return max(scores, key=scores.get)  # return highest scoring cost head
    return "other"



def add_metadata_to_chunks(chunks: list[dict]) -> list[dict]:
    """
    Add cost_head to every chunk.
    Uses keyword matching (fast, free).
    Later we'll upgrade this to LLM mapping for ambiguous cases.
    """
    for chunk in chunks:
        cost_head = map_cost_head_keywords(
            chunk.get("text", ""),
            chunk.get("section_heading", "")
        )
        chunk["cost_head"] = cost_head
    return chunks




def chunk_all_pages(pages: list[dict]) -> list[dict]:

    all_chunks = []
    current_heading = "introduction"
    current_main_section = "introduction"

    for page_data in pages:
        page_metadata = {
            "page_number"   : page_data["page_number"],
            "discom"        : page_data.get("discom", ""),
            "document_type" : page_data.get("document_type", ""),
            "filing_year"   : page_data.get("filing_year", ""),
        }

        # text chunks first
        chunks = chunk_page(
            page_data["text"],
            page_metadata,
            initial_heading=current_heading,
            initial_main_section=current_main_section
        )

        # update heading from text chunks
        if chunks:
            current_heading = chunks[-1]["section_heading"]
            current_main_section = chunks[-1]["main_section"]

        # extend text chunks
        all_chunks.extend(chunks)
        all_chunks = add_metadata_to_chunks(all_chunks)


        # table chunks AFTER — use updated heading
        for table_text in page_data.get("tables_text", []):
            if table_text.strip():
                all_chunks.append({
                    "text"           : table_text,
                    "chunk_type"     : "table",
                    "section_heading": current_heading,
                    "main_section"   : current_main_section,
                    **page_metadata
                })

    return all_chunks

In [17]:
def extract_scanned_page(pdf_path: str, page_num: int) -> dict:
    """
    Extract text from a scanned PDF page using OCR.

    Steps:
    1. Render PDF page as high-res image (300 DPI)
    2. Run Tesseract OCR on the image
    3. Return extracted text

    Note: Tables from scanned pages come out as plain text.
    Structured table extraction from scans needs advanced tools
    like AWS Textract or Google Document AI.
    """
    try:
        import pytesseract
    except ImportError:
        return {"text": "", "tables_text": [], "tables_raw": [],
                "page_number": page_num + 1, "page_type": "scanned",
                "error": "pytesseract not installed"}

    doc = fitz.open(pdf_path)
    page = doc[page_num]

    # 300 DPI gives good OCR quality without being too slow
    # 72 DPI is screen resolution → 300/72 ≈ 4.16x scale
    mat = fitz.Matrix(300 / 72, 300 / 72)
    pix = page.get_pixmap(matrix=mat)
    img = Image.open(io.BytesIO(pix.tobytes("png")))
    doc.close()

    # Run OCR — 'eng' for English, add '+hin' if Hindi text exists
    text = pytesseract.image_to_string(img, lang='eng')

    return {
        "text": text,
        "tables_text": [],    # no structured tables from OCR
        "tables_raw": [],
        "page_number": page_num + 1,
        "page_type": "scanned"
    }



def load_pdf(pdf_path: str, doc_metadata: dict, max_pages: int = None) -> list[dict]:
    """
    Load entire PDF. For each page:
    1. Classify (digital or scanned)
    2. Extract with correct extractor
    3. Attach document-level metadata to every page

    Returns list of page dicts, one per page.
    """
    doc = fitz.open(pdf_path)
    total_pages = len(doc)

    # Optionally limit pages for testing
    pages_to_process = min(max_pages, total_pages) if max_pages else total_pages

    print(f"📄 Loading: {pdf_path}")
    print(f"   Total pages: {total_pages} | Processing: {pages_to_process}")
    print()

    all_pages = []

    for i in range(pages_to_process):
        page = doc[i]
        page_type = detect_page_type(page)

        # Route to correct extractor
        if page_type == "digital":
            page_data = extract_digital_page(pdf_path, i)
        else:
            page_data = extract_scanned_page(pdf_path, i)

        # Merge document-level metadata into every page
        # This is critical — every chunk inherits this later
        page_data.update(doc_metadata)

        all_pages.append(page_data)

        # Progress indicator
        icon = "✅" if page_type == "digital" else "⚠️ "
        print(f"  {icon} Page {i+1:3d}/{pages_to_process} | {page_type:<8} | "
              f"{len(page_data['text']):>5} chars | "
              f"{len(page_data.get('tables_raw', []))} tables")

    doc.close()
    print(f"\n✅ Done — {len(all_pages)} pages loaded")
    return all_pages



# PDF_PATH = r"D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf"


# # ── TEST: Load first 20 pages (fast) ──
# if Path(PDF_PATH).exists():
#     pages = load_pdf(PDF_PATH, DOC_METADATA, max_pages=20)
#     print(DOC_METADATA)
#     print(f"\nSample page keys: {list(pages[0].keys())}")
# else:
#     print("⚠️  Add your PDF first")

In [18]:
# load all pages first
pages = load_pdf(PDF_PATH, DOC_METADATA)

# chunk all pages
all_chunks = chunk_all_pages(pages)

# summary
from collections import Counter

print(f"Total chunks     : {len(all_chunks)}")
print(f"Text chunks      : {len([c for c in all_chunks if c['chunk_type'] == 'text'])}")
print(f"Table chunks     : {len([c for c in all_chunks if c['chunk_type'] == 'table'])}")
print()
print("Main sections found:")
main_sections = Counter(c['main_section'] for c in all_chunks)
for section, count in main_sections.most_common():
    print(f"  {count:>3} chunks → {section}")

📄 Loading: D:\python scripts\Generative AI\arise_chatbot\data\petitions\tariff_petition_jusnl.pdf
   Total pages: 66 | Processing: 66

  ✅ Page   1/66 | digital  |   345 chars | 1 tables
  ✅ Page   2/66 | digital  |  2193 chars | 0 tables
  ✅ Page   3/66 | digital  |  2269 chars | 0 tables
  ✅ Page   4/66 | digital  |  1220 chars | 0 tables
  ✅ Page   5/66 | digital  |  5837 chars | 0 tables
  ✅ Page   6/66 | digital  |  2588 chars | 0 tables
  ✅ Page   7/66 | digital  |    26 chars | 0 tables
  ✅ Page   8/66 | digital  |  2345 chars | 0 tables
  ✅ Page   9/66 | digital  |  2884 chars | 0 tables
  ✅ Page  10/66 | digital  |  4154 chars | 1 tables
  ✅ Page  11/66 | digital  |  1737 chars | 2 tables
  ✅ Page  12/66 | digital  |  1706 chars | 1 tables
  ✅ Page  13/66 | digital  |  2396 chars | 0 tables
  ✅ Page  14/66 | digital  |   867 chars | 0 tables
  ✅ Page  15/66 | digital  |  2410 chars | 0 tables
  ✅ Page  16/66 | digital  |  2664 chars | 2 tables
  ✅ Page  17/66 | digital  |  245

## Chunking by position

### Chunking Based on Table of Content (TOC) Search Method

In [ ]:
def extract_toc_from_page(pdf_path: str, toc_pages: list[int]) -> list[dict]:
    """
    Extract TOC manually from visual TOC pages.
    toc_pages: list of page numbers (0-indexed)
    """
    import re

    toc_entries = []

    with pdfplumber.open(pdf_path) as pdf:
        for page_num in toc_pages:
            page = pdf.pages[page_num]
            text = page.extract_text() or ""

            for line in text.splitlines():
                line = line.strip()

                # TOC line pattern:
                # "3.4. Gross Fixed Asset ................... 16"
                # section title + dots + page number at end
                match = re.match(
                    r'^(\d+\.[\d\.]*)\s+(.+?)\s*\.{2,}\s*(\d+)$',
                    line
                )
                if match:
                    section_num = match.group(1)
                    title       = match.group(2).strip()
                    page_num_val = int(match.group(3))

                    toc_entries.append({
                        "section_num": section_num,
                        "title"      : title,
                        "page"       : page_num_val,
                        "full_heading": f"{section_num} {title}"
                    })

    return toc_entries


# TOC is on pages 5 and 6 (0-indexed = 4, 5)
toc = extract_toc_from_page(PDF_PATH, toc_pages=[4, 5])

print(f"TOC entries found: {len(toc)}")
for entry in toc[:]:
    print(f"  Page {entry['page']:>3} | {entry['full_heading']}")
# cell 1: build_full_text
def chunk_using_toc_v2(
    pages: list[dict],
    toc: list[dict],
    doc_metadata: dict
) -> list[dict]:
    """
    Position-based chunking — much more accurate than page-based.
    Chunks text between exact section heading positions.
    """

    full_text, page_offsets = build_full_text(pages)
    section_positions = find_section_positions(full_text, toc, pages)

    all_chunks = []

    for i, section in enumerate(section_positions):

        # start = this section heading position
        start = section["start_pos"]

        # end = next section heading position
        end = section_positions[i + 1]["start_pos"] if i + 1 < len(section_positions) else len(full_text)

        # extract exact section text
        section_text = full_text[start:end].strip()

        if len(section_text) < 50:
            continue

        # find parent section
        section_num  = section["section_num"]
        is_top_level = section_num.count('.') == 1

        if is_top_level:
            main_section = section["full_heading"]
        else:
            parent_prefix = section_num.split('.')[0] + '.'
            parent = next(
                (s["full_heading"] for s in section_positions
                 if s["section_num"] == parent_prefix),
                section["full_heading"]
            )
            main_section = parent

        # find which page this section starts on
        page_num = section["page"]

        all_chunks.append({
            "text"            : section_text,
            "chunk_type"      : "text",
            "section_heading" : section["full_heading"],
            "main_section"    : main_section,
            "page_number"     : page_num,
            "cost_head"       : "other",
            **doc_metadata
        })

    return all_chunks




def build_full_text(pages: list[dict]) -> tuple:
    full_text = ""
    page_offsets = {}

    for page_data in pages:
        page_offsets[page_data["page_number"]] = len(full_text)
        full_text += page_data.get("text", "") + "\n"

    return full_text, page_offsets

# cell 2: find_section_positions
def find_section_positions(full_text: str, toc: list[dict], pages: list[dict]) -> list[dict]:
    import re

    # skip TOC pages
    toc_end_offset = 0
    for page_data in pages:
        if page_data["page_number"] <= 7:
            toc_end_offset += len(page_data.get("text", "")) + 1

    content_text = full_text[toc_end_offset:]
    section_positions = []

    for entry in toc:
        section_num  = entry["section_num"]

        # match ONLY at start of line — prevents "3.3.5." matching "3.5."
        pattern = r'(?:^|\n)' + re.escape(section_num) + r'[\s\.]'
        match   = re.search(pattern, content_text)

        if match:
            # adjust position to skip the newline character
            pos = match.start()
            if content_text[pos] == '\n':
                pos += 1

            section_positions.append({
                "section_num" : section_num,
                "full_heading": entry["full_heading"],
                "start_pos"   : toc_end_offset + pos,
                "page"        : entry["page"]
            })
        else:
            print(f"⚠️  Not found: {entry['full_heading']}")

    section_positions.sort(key=lambda x: x["start_pos"])
    return section_positions

toc_chunks_v2 = chunk_using_toc_v2(pages, toc, DOC_METADATA)

om_chunk = next(
    (c for c in toc_chunks_v2
     if c['section_heading'] == '3.5. Operation and Maintenance Expenses'),
    None
)

if om_chunk:
    print(f"✅ Found 3.5. O&M section")
    print(f"   Length : {len(om_chunk['text'])} chars")
    print(f"   Preview: {om_chunk['text'][:400]}")

# !pip install -U langchain-text-splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size    = 1000,
    chunk_overlap = 100,   # overlap preserves context at boundaries
    separators    = ["\n\n", "\n", ".", " "]
)


def split_large_chunks(chunks: list[dict], max_size: int = 1500) -> list[dict]:
    """
    Split large chunks but NEVER split inside a table.
    Tables are kept as complete units.
    """
    final_chunks = []

    for chunk in chunks:
        if len(chunk["text"]) <= max_size:
            final_chunks.append(chunk)
            continue

        # split text at table boundaries first
        # tables start with "Table N:" pattern
        import re

        # split on table boundaries — keep tables intact
        parts = re.split(r'(?=Table \d+:)', chunk["text"])

        current_text = ""
        sub_index    = 0

        for part in parts:
            is_table = part.strip().startswith("Table")

            if is_table:
                # save current text chunk if exists
                if current_text.strip() and len(current_text.strip()) > 50:
                    if len(current_text) > max_size:
                        # split large text portion only
                        splits = splitter.split_text(current_text)
                        for split in splits:
                            new_chunk = chunk.copy()
                            new_chunk["text"] = split
                            new_chunk["sub_chunk_index"] = sub_index
                            final_chunks.append(new_chunk)
                            sub_index += 1
                    else:
                        new_chunk = chunk.copy()
                        new_chunk["text"] = current_text.strip()
                        new_chunk["sub_chunk_index"] = sub_index
                        final_chunks.append(new_chunk)
                        sub_index += 1
                    current_text = ""

                # table goes as its own complete chunk — never split
                new_chunk = chunk.copy()
                new_chunk["text"]           = part.strip()
                new_chunk["chunk_type"]     = "table"
                new_chunk["sub_chunk_index"] = sub_index
                final_chunks.append(new_chunk)
                sub_index += 1

            else:
                current_text += part

        # save remaining text
        if current_text.strip() and len(current_text.strip()) > 50:
            if len(current_text) > max_size:
                splits = splitter.split_text(current_text)
                for split in splits:
                    new_chunk = chunk.copy()
                    new_chunk["text"] = split
                    new_chunk["sub_chunk_index"] = sub_index
                    final_chunks.append(new_chunk)
                    sub_index += 1
            else:
                new_chunk = chunk.copy()
                new_chunk["text"] = current_text.strip()
                new_chunk["sub_chunk_index"] = sub_index
                final_chunks.append(new_chunk)

    return final_chunks


# test
toc_chunks_v2_split = split_large_chunks(toc_chunks_v2, max_size=1500)

print(f"Before: {len(toc_chunks_v2)} chunks")
print(f"After : {len(toc_chunks_v2_split)} chunks")

# verify no table was split
table_chunks = [c for c in toc_chunks_v2_split if c['chunk_type'] == 'table']
print(f"Table chunks: {len(table_chunks)}")

# check O&M section
om_chunks = [
    c for c in toc_chunks_v2_split
    if c['section_heading'] == '3.5. Operation and Maintenance Expenses'
]
print(f"\n3.5 O&M split into {len(om_chunks)} chunks:")
for c in om_chunks:
    print(f"  {c['chunk_type']:<6} | {len(c['text']):>5} chars | {c['text'][:80]}")
toc_chunks_v2_split
import re

def annotate_chunk(chunk: dict) -> dict:
    """
    Add sub-section annotations to chunk text.
    Extracts all paragraph numbers like 3.7.1, 3.7.2 from text.
    """
    text = chunk["text"]

    # find all sub-section numbers in text
    # pattern: digit.digit.digit like 3.7.1, 3.7.2, 4.5.3
    sub_sections = re.findall(r'\d+\.\d+\.\d+\.?', text)

    # deduplicate while preserving order
    seen = set()
    unique_subs = []
    for s in sub_sections:
        s = s.rstrip('.')
        if s not in seen:
            seen.add(s)
            unique_subs.append(s)

    if unique_subs:
        # add annotation prefix to text
        annotation = (
            f"[Document: {chunk.get('discom', '')} | "
            f"Section: {chunk['section_heading']} | "
            f"Paragraphs: {', '.join(unique_subs)}]\n\n"
        )
        chunk["text"]        = annotation + text
        chunk["sub_sections"] = unique_subs  # also store in metadata

    return chunk


def annotate_all_chunks(chunks: list[dict]) -> list[dict]:
    """Apply annotation to all text chunks. Skip table chunks."""
    annotated = []
    for chunk in chunks:
        if chunk["chunk_type"] == "text":
            chunk = annotate_chunk(chunk)
        annotated.append(chunk)
    return annotated


# apply
toc_chunks_annotated = annotate_all_chunks(toc_chunks_v2_split.copy())

# verify
interest_chunk = next(
    (c for c in toc_chunks_annotated
     if c['section_heading'] == '3.7. Interest and Loan capital'),
    None
)

if interest_chunk:
    print(interest_chunk['text'][:400])
    print()
    print(f"Sub-sections: {interest_chunk.get('sub_sections', [])}")

TOC entries found: 57
  Page   8 | 1. Introduction
  Page   8 | 1.1. Background
  Page   9 | 1.2. Profile of JUSNL
  Page  11 | 1.3. Procedural History
  Page  12 | 1.4. Rationale for filing of Instant Petition
  Page  12 | 1.5. Contents of the Petition
  Page  13 | 2. Overall Approach and Provision of Law
  Page  13 | 2.1. Present Approach
  Page  13 | 2.2. Data and information sources
  Page  13 | 2.3. Provision of Law
  Page  15 | 3. Provisional True up For FY 2023-24
  Page  15 | 3.1. Preamble
  Page  15 | 3.2. Basis of True up
  Page  15 | 3.3. Capital Expenditure, Capitalization and CWIP
  Page  16 | 3.4. Gross Fixed Asset
  Page  17 | 3.5. Operation and Maintenance Expenses
  Page  21 | 3.6. Depreciation
  Page  23 | 3.7. Interest and Loan capital
  Page  23 | 3.8. Return on Equity
  Page  24 | 3.9. Interest on Working Capital
  Page  25 | 3.10. Non-Tariff Income
  Page  26 | 3.11. Tax on income
  Page  26 | 3.12. Incentive for Target Availability
  Page  28 | 3.13. Revenue from

In [35]:
len(toc_chunks_v2[1]['text'])
len(toc_chunks_v2[0]['text'])

5005

In [36]:
toc_chunks_v2[1]['text']


'1.2. Profile of JUSNL\n1.2.1. JUSNL is engaged primarily in the business of transmission of electricity. It has\nbeen vested with the transmission assets, interest in property, rights and liabilities\n9 | P a ge\nof the erstwhile JSEB necessary for the business of transmission in the state of\nJharkhand.\n1.2.2. JUSNL has been given the status of a Transmission Licensee as per Section 14 of\nthe Electricity Act 2003, to fulfill the obligations of the Transmission Licensee as\nmandated under the provisions of “The Jharkhand State Electricity Reforms\nRevised Transfer Scheme, 2015” and the Electricity Act, 2003.\n1.2.3. The Jharkhand State Electricity Reforms Revised Transfer Scheme, 2015 details\nout the following for the transmission business of JUSNL under Schedule- ‘A’\nTransmission Undertaking:\nPart I: Transmission Assets, General Assets, Miscellaneous\nPart II: Aggregate Assets and Liabilities\nPart III: Functions and Duties of JUSNL\n1.2.4. At the time of creation of JSEB (erstw

In [20]:
def chunk_using_toc(pdf_path: str, toc: list[dict],
                    pages: list[dict], doc_metadata: dict) -> list[dict]:
    """
    Use TOC as map — chunk content between section boundaries.
    """
    all_chunks = []

    for i, entry in enumerate(toc):

        # find start and end page for this section
        start_page = entry["page"]          # 1-indexed from TOC

        # end page = next section's start page
        if i + 1 < len(toc):
            end_page = toc[i + 1]["page"]
        else:
            end_page = len(pages) + 1       # last section goes to end

        # collect text from all pages in this section's range
        section_text = ""
        section_tables = []

        for page_data in pages:
            pg = page_data["page_number"]   # 1-indexed
            if start_page <= pg < end_page:
                section_text += "\n" + page_data.get("text", "")
                section_tables.extend(page_data.get("tables_text", []))

        # determine main section
        section_num = entry["section_num"]
        is_top_level = section_num.count('.') == 1  # "3." not "3.1."

        page_metadata = {
            "page_number"    : start_page,
            "section_heading": entry["full_heading"],
            "main_section"   : entry["full_heading"] if is_top_level
                               else _find_parent(toc, i),
            "cost_head"      : "other",
            **doc_metadata
        }

        # text chunk
        if section_text.strip() and len(section_text.strip()) > 100:
            all_chunks.append({
                "text"      : section_text.strip(),
                "chunk_type": "text",
                **page_metadata
            })

        # table chunks
        for table_text in section_tables:
            if table_text.strip():
                all_chunks.append({
                    "text"      : table_text,
                    "chunk_type": "table",
                    **page_metadata
                })

    return all_chunks


def _find_parent(toc: list[dict], current_idx: int) -> str:
    """
    Find parent section for a subsection.
    For 3.4. → find most recent 3. entry above it.
    """
    current_num = toc[current_idx]["section_num"]
    parent_prefix = current_num.split('.')[0] + '.'

    for i in range(current_idx - 1, -1, -1):
        if toc[i]["section_num"] == parent_prefix:
            return toc[i]["full_heading"]

    return toc[current_idx]["full_heading"]

In [21]:
toc_chunks = chunk_using_toc(PDF_PATH, toc, pages, DOC_METADATA)

from collections import Counter
print(f"Total chunks  : {len(toc_chunks)}")
print(f"Text chunks   : {len([c for c in toc_chunks if c['chunk_type'] == 'text'])}")
print(f"Table chunks  : {len([c for c in toc_chunks if c['chunk_type'] == 'table'])}")
print()
print("Sections:")
for entry in toc[:]:
    matching = [c for c in toc_chunks
                if c['section_heading'] == entry['full_heading']]
    print(f"  {len(matching)} chunks → {entry['full_heading']}")

Total chunks  : 101
Text chunks   : 37
Table chunks  : 64

Sections:
  0 chunks → 1. Introduction
  1 chunks → 1.1. Background
  2 chunks → 1.2. Profile of JUSNL
  3 chunks → 1.3. Procedural History
  0 chunks → 1.4. Rationale for filing of Instant Petition
  2 chunks → 1.5. Contents of the Petition
  0 chunks → 2. Overall Approach and Provision of Law
  0 chunks → 2.1. Present Approach
  0 chunks → 2.2. Data and information sources
  1 chunks → 2.3. Provision of Law
  0 chunks → 3. Provisional True up For FY 2023-24
  0 chunks → 3.1. Preamble
  0 chunks → 3.2. Basis of True up
  1 chunks → 3.3. Capital Expenditure, Capitalization and CWIP
  3 chunks → 3.4. Gross Fixed Asset
  6 chunks → 3.5. Operation and Maintenance Expenses
  4 chunks → 3.6. Depreciation
  0 chunks → 3.7. Interest and Loan capital
  2 chunks → 3.8. Return on Equity
  2 chunks → 3.9. Interest on Working Capital
  2 chunks → 3.10. Non-Tariff Income
  0 chunks → 3.11. Tax on income
  3 chunks → 3.12. Incentive for Targ

In [22]:
# toc_chunks

## Collection Creation

In [ ]:
# !pip install qdrant-client==1.9.0

In [24]:
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams

# ============================================================
# METHOD 1: WITHOUT DOCKER (local file mode)
# - No setup needed
# - Data saves directly to your disk
# - Connect via file path
# ============================================================

client = QdrantClient(path="data/qdrant_store")


# ============================================================
# METHOD 2: WITH DOCKER
# First run this command in terminal:
# docker run -p 6333:6333 \
#   -v D:/qdrant_storage:/qdrant/storage \
#   qdrant/qdrant
#
# Then connect from Python:
# ============================================================

# client = QdrantClient(host="localhost", port=6333)


# ============================================================
# COLLECTION SETUP (same for both methods)
# Run this ONCE only — skip if collection already exists
# ============================================================

COLLECTION_NAME = "petition_chunks"
VECTOR_SIZE = 1024  # BAAI/bge-large-en-v1.5 output size

def create_collection(client):
    # check if collection already exists
    existing = [c.name for c in client.get_collections().collections]

    if COLLECTION_NAME not in existing:
        client.create_collection(
            collection_name=COLLECTION_NAME,
            vectors_config=VectorParams(
                size=VECTOR_SIZE,
                distance=Distance.COSINE
            )
        )
        print(f"✅ Collection '{COLLECTION_NAME}' created")
    else:
        print(f"ℹ️  Collection '{COLLECTION_NAME}' already exists — skipping")

create_collection(client)

RuntimeError: Storage folder data/qdrant_store is already accessed by another instance of Qdrant client. If you require concurrent access, use Qdrant server instead.

## Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model...")
embedder = SentenceTransformer('BAAI/bge-large-en-v1.5')
print(f"✅ Model loaded | Vector size: {embedder.get_sentence_embedding_dimension()}")

## Upserting the emebedding data

In [25]:
from qdrant_client.models import PointStruct

def upsert_chunks(client, chunks: list[dict], embedder, batch_size: int = 32):
    """
    Embed all chunks and store in Qdrant.
    Processes in batches to avoid memory issues.
    """
    print(f"Upserting {len(chunks)} chunks to Qdrant...")

    # step 1: embed all texts at once in batches
    texts = [chunk["text"] for chunk in chunks]
    embeddings = embedder.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    # step 2: create points and upsert in batches
    for i in range(0, len(chunks), batch_size):
        batch_chunks     = chunks[i : i + batch_size]
        batch_embeddings = embeddings[i : i + batch_size]

        points = []
        for j, (chunk, embedding) in enumerate(zip(batch_chunks, batch_embeddings)):
            points.append(
                PointStruct(
                    id      = i + j,           # unique id
                    vector  = embedding.tolist(),
                    payload = chunk            # entire chunk dict as payload
                )
            )

        # step 3: upsert batch to qdrant
        client.upsert(
            collection_name = COLLECTION_NAME,
            points          = points
        )

        print(f"  Upserted {min(i + batch_size, len(chunks))}/{len(chunks)} chunks")

    print(f"✅ Done — {len(chunks)} chunks stored in Qdrant")


# run it
def upsert_chunks_if_empty(client, chunks, embedder, batch_size=32):
    """
    Only upsert if collection is empty.
    Skips if data already exists.
    """
    info = client.get_collection(COLLECTION_NAME)

    if info.points_count > 0:
        print(f"ℹ️  Collection already has {info.points_count} vectors — skipping upsert")
        return

    print(f"Collection is empty — upserting {len(chunks)} chunks...")
    upsert_chunks(client, chunks, embedder, batch_size)


# replace your upsert call with this
upsert_chunks_if_empty(client, toc_chunks, embedder)

NameError: name 'client' is not defined

In [ ]:
# check collection info
info = client.get_collection(COLLECTION_NAME)
print(f"Total vectors stored : {info.points_count}")
print(f"Vector size          : {info.config.params.vectors.size}")
print(f"Distance metric      : {info.config.params.vectors.distance}")

## Retriever

In [ ]:
from qdrant_client.models import Filter, FieldCondition, MatchValue

def retrieve(
    query: str,
    client,
    embedder,
    top_k: int = 5,
    filters: dict = None
):
    """
    Retrieve relevant chunks for a query.
    Optionally filter by metadata before vector search.
    """
    # step 1: embed the query
    query_vector = embedder.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    # step 2: build metadata filter if provided
    qdrant_filter = None
    if filters:
        conditions = [
            FieldCondition(
                key=key,
                match=MatchValue(value=value)
            )
            for key, value in filters.items()
        ]
        qdrant_filter = Filter(must=conditions)

    # step 3: search qdrant
    results = client.search(
        collection_name = COLLECTION_NAME,
        query_vector    = query_vector,
        limit           = top_k,
        query_filter    = qdrant_filter,
        with_payload    = True   # return metadata with results
    )

    # step 4: format results
    chunks = []
    for r in results:
        chunk = r.payload
        chunk["score"] = round(r.score, 3)
        chunks.append(chunk)

    return chunks

from sentence_transformers import CrossEncoder

# load reranker model — downloads ~80MB first time
print("Loading reranker...")
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
print("✅ Reranker loaded")


def rerank_chunks(
    query: str,
    chunks: list[dict],
    reranker,
    top_k: int = 3
) -> list[dict]:
    """
    Re-score retrieved chunks by actual relevance to query.
    Returns top_k most relevant chunks.
    """
    if not chunks:
        return []

    # create query-chunk pairs for reranker
    pairs = [(query, chunk["text"]) for chunk in chunks]

    # score each pair
    scores = reranker.predict(pairs)

    # attach reranker score to each chunk
    for chunk, score in zip(chunks, scores):
        chunk["reranker_score"] = round(float(score), 3)

    # sort by reranker score
    reranked = sorted(
        chunks,
        key=lambda x: x["reranker_score"],
        reverse=True
    )

    return reranked[:top_k]

In [ ]:
# test 1: no filter
results = retrieve(
    query  = "What is ARR for FY 2025-26?",
    client = client,
    embedder = embedder,
    top_k  = 3
)

for i, r in enumerate(results):
    print(f"\nResult {i+1} | Score: {r['score']} | Page: {r['page_number']}")
    print(f"  Section : {r['section_heading']}")
    print(f"  Text    : {r['text'][:200]}")

print()

In [ ]:
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv()


def format_context(chunks: list[dict]) -> str:
    """
    Format retrieved chunks into clean context string for LLM.
    Each chunk shows section, page, and content.
    """
    context_parts = []
    for i, chunk in enumerate(chunks):
        context_parts.append(
            f"[Source {i+1} | Page {chunk['page_number']} | "
            f"Section: {chunk['section_heading']}]\n"
            f"{chunk['text']}"
        )
    return "\n\n---\n\n".join(context_parts)

# First, ensure you have it installed: pip install langchain-google-genai
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))

# This will now work:
response = llm.invoke("Hello")
print(response.content)

# First, ensure you have it installed: pip install langchain-google-genai

# from langchain_google_genai import ChatGoogleGenerativeAI

# llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))

# # This will now work:
# response = llm.invoke("Hello")
# print(response.content)

def ask(
    question: str,
    client,
    embedder,
    llm,
    top_k: int = 5,
    filters: dict = None
) -> dict:

    # step 1: retrieve
    chunks = retrieve(
        query    = question,
        client   = client,
        embedder = embedder,
        top_k    = top_k,
        filters  = filters
    )
    chunks = rerank_chunks(question, chunks, reranker, top_k)

    if not chunks:
        return {"answer": "No relevant chunks found.", "citations": []}

    # step 2: format context
    context = format_context(chunks)

    # step 3: prompt
    prompt = f"""You are an expert Indian power sector regulatory analyst.

You are given context extracted from a tariff petition filed by JUSNL before JSERC.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Answer ONLY using the context provided
2. Include exact numbers and figures
3. Cite source at end like: [Source X | Page Y | Section Z]
4. If not found say: "Information not found in provided context"
5. Keep answer concise and precise

ANSWER:"""

    # step 4: call ChatGroq
    response = llm.invoke([HumanMessage(content=prompt)])
    answer   = response.content.strip()

    # step 5: citations
    citations = [
        {
            "source"  : i + 1,
            "page"    : c["page_number"],
            "section" : c["section_heading"],
            "score"   : c["score"]
        }
        for i, c in enumerate(chunks)
    ]

    return {
        "question" : question,
        "answer"   : answer,
        "citations": citations
    }


# test
result = ask(
    question = "What is the Annual Revenue Requirement for FY 2025-26?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    top_k    = 5,
    filters  = {"main_section": "5. ARR for the FY 2025-26"}
)

print(f"QUESTION :\n{result['question']}")
print(f"\nANSWER   :\n{result['answer']}")
print(f"\nCITATIONS:")
for c in result['citations']:
    print(f"  Source {c['source']} | Page {c['page']} | {c['section']} | Score: {c['score']}")

d:\python scripts\Generative AI\Arise\arise\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# find what section headings exist for capitalization
cap_chunks = [
    c for c in toc_chunks
    if "capital" in c['section_heading'].lower()
    and "5." in c['section_heading']
]

for c in cap_chunks:
    print(c['section_heading'])

In [ ]:
# find all unique section headings in main section 5
section_5_headings = set(
    c['section_heading']
    for c in toc_chunks
    if c['main_section'] == "5. ARR for the FY 2025-26"
)

for h in sorted(section_5_headings):
    print(h)

In [ ]:
# check what's in 5.3 chunk
chunk = next(c for c in toc_chunks if c['section_heading'] == '5.3. Gross Fixed Asset')
print(chunk['text'][:500])

In [ ]:
def rewrite_query(question: str, llm) -> str:
    """
    Rewrite user query to be more precise for regulatory document retrieval.
    Resolves ambiguous terms like 'approved', 'projected', 'actual'.
    """
    prompt = f"""You are an Indian power sector regulatory expert.

    Rewrite the following user query to be more precise for searching a tariff petition document.

    Rules:
    1. Replace ambiguous terms:
    - "approved" → "previously approved by regulator"
    - "projected" → "estimated/claimed by Discom for future year"
    - "actual" → "audited actual figures"
    2. Add relevant regulatory terms if missing
    3. IMPORTANT: This is a TRANSMISSION company petition — use "Transmission Licensee" not "Discom"
    4. Keep it concise — one sentence only
    5. Return ONLY the rewritten query, nothing else

    Original query: {question}

    Rewritten query:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    return response.content.strip()



def route_query(question: str, llm, toc: list[dict]) -> list[str]:
    """
    Given a query, LLM decides which main sections to search.
    Returns list of relevant main_section values.
    """
    # build section list from TOC
    main_sections = list(set(
        entry["full_heading"]
        for entry in toc
        if entry["section_num"].count(".") == 1  # top level only
    ))

    sections_str = "\n".join(f"- {s}" for s in main_sections)

    prompt = f"""You are an Indian power sector regulatory expert.

Given this user query, which sections of a tariff petition are most relevant?

Available sections:
{sections_str}

User query: {question}

Instructions:
1. Return ONLY the section names that are relevant
2. Return as comma separated list
3. If query spans multiple years return all relevant sections
4. Return maximum 3 sections

Relevant sections:"""

    response = llm.invoke([HumanMessage(content=prompt)])

    # parse response into list
    raw = response.content.strip()
    sections = [s.strip() for s in raw.split(",")]
    # with this — match actual main_section values from chunks

    actual_main_sections = list(set(
        c["main_section"] for c in toc_chunks
    ))

    valid_sections = []
    for s in sections:
        for actual in actual_main_sections:
            if s.lower() in actual.lower():
                valid_sections.append(actual)
                break

    return valid_sections if valid_sections else None


def ask(
    question: str,
    client,
    embedder,
    llm,
    toc: list[dict],          # ← add toc parameter
    top_k: int = 8,
    filters: dict = None,
    rewrite: bool = True
) -> dict:

    # step 0: rewrite query
    rewritten_query = question
    if rewrite:
        rewritten_query = rewrite_query(question, llm)

    # step 0.5: auto route if no filter provided
    if filters is None:
        sections = route_query(question, llm, toc)
        print(f"Auto-routed to: {sections}")

        if sections:
            # retrieve from each section separately
            all_chunks = []
            for section in sections:
                chunks = retrieve(
                    query    = rewritten_query,
                    client   = client,
                    embedder = embedder,
                    top_k    = 3,
                    filters  = {"main_section": section}
                )
                chunks = rerank_chunks(question, chunks, reranker, top_k=3)
                all_chunks.extend(chunks)

            # sort by score
# correct — sorting by reranker score
            chunks = sorted(
                all_chunks,
                key=lambda x: x.get("reranker_score", x["score"]),
                reverse=True
            )[:top_k]
        else:
            chunks = retrieve(
                query    = rewritten_query,
                client   = client,
                embedder = embedder,
                top_k    = top_k,
                filters  = None
            )
            chunks = rerank_chunks(rewritten_query, chunks, reranker, top_k)
    else:
        chunks = retrieve(
            query    = rewritten_query,
            client   = client,
            embedder = embedder,
            top_k    = top_k,
            filters  = filters
        )
        chunks = rerank_chunks(rewritten_query, chunks, reranker, top_k)

    if not chunks:
        return {"answer": "No relevant chunks found.", "citations": []}

    # step 2: format context
    context = format_context(chunks)

    # step 3: prompt
    prompt = f"""You are an expert Indian power sector regulatory analyst.

You are given context extracted from a tariff petition filed by JUSNL before JSERC.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Answer ONLY using the context provided
2. Include exact numbers and figures
3. Cite source at end like: [Source X | Page Y | Section Z]
4. If not found say: "Information not found in provided context"
5. Keep answer concise and precise

ANSWER:"""

    # step 4: call LLM
    response = llm.invoke([HumanMessage(content=prompt)])
    answer   = response.content.strip()

    # step 5: citations
    citations = [
        {
            "source"  : i + 1,
            "page"    : c["page_number"],
            "section" : c["section_heading"],
            "score"   : c["score"]
        }
        for i, c in enumerate(chunks)
    ]

    return {
        "question"        : question,
        "rewritten_query" : rewritten_query,
        "answer"          : answer,
        "citations"       : citations
    }

    # rest stays same...

In [ ]:
result = ask(
    question = "What is the approved ARR for FY 2025-26 for JUSNL?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    top_k    = 5,
    toc      = toc,
    filters  = {"main_section": "5. ARR for the FY 2025-26"},
    rewrite  = True
)

print(f"\nA: {result['answer']}")

In [ ]:
result = ask(
    question = "What is the ARR claimed by JUSNL for FY 2025-26?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    top_k    = 5,
    toc      = toc,
    filters  = {"main_section": "5. ARR for the FY 2025-26"},
    rewrite  = True
)

print(f"Original : What is the ARR claimed by JUSNL for FY 2025-26?")
print(f"Rewritten: {result.get('rewritten_query', 'not stored')}")
print(f"\nA: {result['answer']}")

In [ ]:
# retrieve separately for each year then combine
chunks_fy24 = retrieve(
    query    = "capitalization FY 2023-24 approved actual",
    client   = client,
    embedder = embedder,
    top_k    = 3,
    filters  = {"main_section": "3. Provisional True up For FY 2023-24"}
)

chunks_fy25 = retrieve(
    query    = "capitalization FY 2024-25 approved projected",
    client   = client,
    embedder = embedder,
    top_k    = 3,
    filters  = {"main_section": "4. APR for the FY 2024-25"}
)

chunks_fy26 = retrieve(
    query    = "capitalization FY 2025-26 approved projected",
    client   = client,
    embedder = embedder,
    top_k    = 3,
    filters  = {"main_section": "5. ARR for the FY 2025-26"}
)

# combine all chunks
all_chunks = chunks_fy24 + chunks_fy25 + chunks_fy26
context = format_context(all_chunks)
print(f"Total chunks: {len(all_chunks)}")
print(context[:500])

In [ ]:
sections = route_query(
    question = "Compare approved vs actual capitalization across FY24, FY25 and FY26",
    llm      = llm,
    toc      = toc
)
print(sections)

In [ ]:
result = ask(
    question = "Compare approved vs actual capitalization across FY24, FY25 and FY26",
    client   = client,
    embedder = embedder,
    llm      = llm,
    toc      = toc,
    rewrite  = True
)

print(f"A: {result['answer']}")

In [ ]:
def ask_comparison(
    question: str,
    client,
    embedder,
    llm,
    toc: list[dict],
    sections: list[str],
    per_section_k: int = 5
) -> dict:
    """
    Special handler for comparison queries across multiple years.
    Retrieves more chunks per section with targeted query.
    """
    rewritten_query = rewrite_query(question, llm)

    all_chunks = []
    for section in sections:
        chunks = retrieve(
            query    = "capital expenditure capitalization approved actual projected",
            client   = client,
            embedder = embedder,
            top_k    = per_section_k,
            filters  = {"main_section": section}
        )

        all_chunks.extend(chunks)

    # deduplicate by page number
    seen_pages = set()
    unique_chunks = []
    for c in all_chunks:
        if c["page_number"] not in seen_pages:
            seen_pages.add(c["page_number"])
            unique_chunks.append(c)

    context = format_context(unique_chunks)

    prompt = f"""You are an expert Indian power sector regulatory analyst.

CONTEXT:
{context}

QUESTION: {question}

INSTRUCTIONS:
1. Compare values across ALL three years FY2023-24, FY2024-25, FY2025-26
2. Show approved vs actual/projected for each year in a clear format
3. Use exact numbers from context
4. If a year is missing say "not found"
5. Cite sources

ANSWER:"""

    response = llm.invoke([HumanMessage(content=prompt)])

    return {
        "question": question,
        "answer"  : response.content.strip(),
        "citations": [{"page": c["page_number"], "section": c["section_heading"]}
                     for c in unique_chunks]
    }


# test
result = ask_comparison(
    question = "Compare approved vs actual capitalization across FY24, FY25 and FY26",
    client   = client,
    embedder = embedder,
    llm      = llm,
    toc      = toc,
    sections = [
        "3. Provisional True up For FY 2023-24",
        "4. APR for the FY 2024-25",
        "5. ARR for the FY 2025-26"
    ]
)

print(result['answer'])

In [ ]:
# after retrieving chunks add this line
# chunks = rerank_chunks(question, chunks, reranker, top_k=3)

In [ ]:
result = ask(
    question = "What was actual employee cost for FY 2023-24?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    toc      = toc,
    rewrite  = True
)

print(f"A: {result['answer']}")
for c in result['citations']:
    print(f"  Page {c['page']} | {c['section']}")

In [ ]:
result = ask(
    question = "What was actual employee cost for FY 2023-24?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    toc      = toc,
    rewrite  = True
)

print(f"A: {result['answer']}")
for c in result['citations']:
    print(f"  Page {c['page']} | {c['section']}")

In [ ]:
# search directly for page 17
from qdrant_client.models import Filter, FieldCondition, MatchValue

results = client.scroll(
    collection_name = COLLECTION_NAME,
    scroll_filter = Filter(
        must=[
            FieldCondition(
                key="page_number",
                match=MatchValue(value=17)
            )
        ]
    ),
    with_payload = True,
    limit = 10
)

for point in results[0]:
    print(f"Page: {point.payload['page_number']}")
    print(f"Section: {point.payload['section_heading']}")
    print(f"Text preview: {point.payload['text'][:150]}")
    print("---")

In [ ]:
result = ask(
    question = "What was actual employee cost for FY 2023-24?",
    client   = client,
    embedder = embedder,
    llm      = llm,
    toc      = toc,
    rewrite  = True
)

print(f"A: {result['answer']}")
for c in result['citations']:
    print(f"  Page {c['page']} | {c['section']}")

In [ ]:
# test router directly
sections = route_query(
    question = "What was actual employee cost for FY 2023-24?",
    llm      = llm,
    toc      = toc
)
print(f"Raw sections: {sections}")

In [ ]:
# debug router - see raw LLM response
def debug_route_query(question: str, llm, toc: list[dict]):
    main_sections = list(set(
        entry["full_heading"]
        for entry in toc
        if entry["section_num"].count(".") == 1
    ))

    sections_str = "\n".join(f"- {s}" for s in main_sections)

    print("Available sections:")
    print(sections_str)
    print()

    prompt = f"""You are an Indian power sector regulatory expert.

Given this user query, which sections of a tariff petition are most relevant?

Available sections:
{sections_str}

User query: {question}

Instructions:
1. Return ONLY the section names that are relevant
2. Return as comma separated list
3. If query spans multiple years return all relevant sections
4. Return maximum 3 sections

Relevant sections:"""

    response = llm.invoke([HumanMessage(content=prompt)])
    print(f"Raw LLM response: '{response.content}'")

debug_route_query(
    "What was actual employee cost for FY 2023-24?",
    llm,
    toc
)